In [2]:
import pandas as pd
from pathlib import Path

import load_data

In [ ]:
hl_hype = load_data.get_md(
    'HYPERLIQUID',
    'HYPE-USD-PERP',
    'book')
    
bin_hype = load_data.get_md(
    'BINANCE_FUTURES',
    'HYPE-USDT-PERP',
    'book')

In [ ]:
hl_hype.head()

In [ ]:
b_1min = (
    bin_hype[["mid", "bid", 'ask', 'bid_size', 'ask_size', 'imbalance', 'microprice']]
    .resample("60000ms")
    .last()
    .ffill()
    .rename(columns={"mid": "bin_mid", "bid": "bin_bid", 
                    'bid_size':'bid_size_bin', 'ask_size':'ask_size_bin',
                    'imbalance':'imbalance_bin', 'microprice':'microprice_bin'})
)

h_1min = (
    hl_hype[["mid", "bid", 'bid_size', 'ask_size', 'imbalance', 'microprice']]
    .resample("60000ms")
    .last()
    .ffill()
    .rename(columns={"mid": "hl_mid", "bid": "hl_bid",
                    'bid_size':'bid_size_hl', 'ask_size':'ask_size_hl',
                    'imbalance':'imbalance_hl', 'microprice':'microprice_hl'})
)

# Merge them on their timestamp indexes
df = b_1min.join(h_1min, how="inner")
df.tail()

In [ ]:
df['microprice_diff'] = df['microprice_hl'] - df['microprice_bin']
df["mid_diff"] = (
    df["hl_mid"]
    - df["bin_mid"]
)

df["reference_mid"] = (
    df["hl_mid"]
    + df["bin_mid"]
) / 2

df["mid_diff_bps"] = (
    df["hl_mid"]
    - df["bin_mid"]
) / df["reference_mid"] * 10_000

In [ ]:
from spread_cost import plot_pnl, cost_breakdown

ref        = (df.hl_mid + df.bin_mid) / 2
spread_bps = (df.hl_mid - df.bin_mid) / ref * 1e4

fig, res = plot_pnl(spread_bps[-2_000:], taker=0.05, maker=0.95,
                    stop_bps=12, horizon_bars=600)   # 600 bars = 60s at 100ms

In [ ]:
df.head()

## Markov-Switching Latent Fair-Value Model

Start with the observed price decomposition:

$$
p_t = m_t + s_t + \epsilon_t
$$

where:

- $p_t$ = observed market price
- $m_t$ = latent efficient / fair price
- $s_t$ = temporary, predictable microstructure distortion
- $\epsilon_t$ = residual observation noise

### Efficient-price process

Model the latent efficient price as a random walk:

$$
m_t = m_{t-1} + \eta_t
$$

where $\eta_t$ represents innovations to the true efficient price.

### Microstructure distortion

Let the temporary distortion depend on both its previous value and current market-microstructure information:

$$
s_t
=
\rho_{z_t}s_{t-1}
+
\beta_{z_t}^{\top}x_t
+
\nu_t
$$

where:

- $z_t$ = latent market regime
- $\rho_{z_t}$ = persistence of temporary price distortion in regime $z_t$
- $\beta_{z_t}$ = regime-dependent weights on market-microstructure variables
- $x_t$ = vector of observable order-book / order-flow features
- $\nu_t$ = innovation to the temporary distortion

A simple initial feature vector could be:

$$
x_t =
\begin{bmatrix}
\text{microprice} - \text{midprice} \\
\text{order-flow imbalance} \\
\text{trade imbalance} \\
\text{bid-ask spread} \\
\text{depth imbalance}
\end{bmatrix}
$$

### Markov regime process

Assume the latent regime follows a Markov process:

$$
z_t \sim \operatorname{Markov}(P)
$$

where:

$$
P_{ij}
=
P(z_t = j \mid z_{t-1} = i)
$$

Possible regimes may correspond to states such as:

1. Balanced / normal market
2. Sustained buy pressure
3. Sustained sell pressure
4. Low-liquidity / stressed market

The regimes do not necessarily need to be manually labelled beforehand; they can be inferred statistically and interpreted afterwards.

### Fair-value estimate

The model produces an estimate of the current temporary distortion:

$$
\hat{s}_t
=
E[s_t \mid \mathcal{F}_t]
$$

and therefore the estimated latent efficient price is:

$$
\boxed{
\hat{m}_t = p_t - \hat{s}_t
}
$$

The purpose of the model is therefore to use observable microstructure information to separate temporary price pressure from movements in the underlying efficient price.

### Research progression

A natural progression for the research is:

1. **Midprice benchmark**
2. **Microprice benchmark**
3. **Linear microstructure fair-value model**
4. **Regime-dependent / Markov model**
5. **Full Markov-switching state-space model**

Each increase in complexity should only be retained if it improves out-of-sample estimates of the latent fair price or subsequent price movement.

In [ ]:
import fv_models as fv
importlib.reload(fv)        # so edits to the module take effect
 
f = fv.build_features(df, basis_window=120, vol_window=60)
print(f.shape)
f.tail(3)

In [ ]:
COST_BPS = 4.92             # from cost_breakdown() earlier in the notebook
 
_ = fv.magnitude_check(f, cost_bps=COST_BPS)

In [ ]:
res_by_h = {}
for h in (5, 15, 30, 60):
    res_by_h[h] = fv.run_ladder(f, horizon=h, cost_bps=COST_BPS, n_splits=5)
    print(f"\n===== horizon = {h} bars =====")
    print(res_by_h[h]["table"].to_string())
 
HORIZON = 30                # pick one to inspect below
res = res_by_h[HORIZON]

In [ ]:
# %% ------------------------------------------------------------------
# CELL 4 - your two actual questions
#
# "both"            trades the incumbent and the model both take
# "incumbent only"  trades the model SKIPS -> if mean_net here is negative,
#                   the moving fair value is avoiding toxic fills
# "<model> only"    trades the model ADDS  -> if mean_net here is positive,
#                   it is finding inefficiency the static rule misses
#
# Either column can be the win. Skipping bad trades and adding good ones
# are different edges and you should not average them together.
# ---------------------------------------------------------------------
print("best rung:", res["best"], "   horizon:", res["horizon"], "bars")
print()
print(res["overlap"].to_string())

In [ ]:
# %% ------------------------------------------------------------------
# CELL 5 - is the regime real, or did the filter fit outliers?
#
# Run this BEFORE believing cell 3. On simulated data with one true regime
# the Markov rung still produces numbers, and these checks are what catch
# it: one state collapses onto a few outliers, giving a sigma ratio of 30
# or 700 and an occupancy of 0.3%.
# ---------------------------------------------------------------------
fv.regime_health(res)
print()
print(fv.regime_summary(res["par"]).to_string())
print()
print(fv.name_the_regime(res).to_string())

In [ ]:
# %% ------------------------------------------------------------------
# CELL 6 - the picture: static vs moving fair value
# ---------------------------------------------------------------------
fvf = fv.fv_frame(f, res).dropna()
tail = fvf.iloc[-2000:]
p_hi = res["probs"].reindex(tail.index).iloc[:, -1]
 
fig, ax = plt.subplots(2, 1, figsize=(13, 7), sharex=True,
                       gridspec_kw={"height_ratios": [3, 1]})
 
ax[0].plot(tail.index, tail["spread"], lw=0.7, color="0.35",
           label="HL - BIN spread")
ax[0].plot(tail.index, tail["fv_static"], lw=1.2, color="tab:red",
           label="static FV (zero)")
ax[0].plot(tail.index, tail["fv_moving"], lw=1.2, color="tab:blue",
           label=f"moving FV (rolling basis)")
ax[0].plot(tail.index, tail["fv_model"], lw=1.0, color="tab:green",
           alpha=0.85, label=f"model FV ({res['best']})")
ax[0].axhspan(-COST_BPS, COST_BPS, color="tab:red", alpha=0.07,
              label=f"round-trip cost band (+/-{COST_BPS:.2f} bps)")
ax[0].set_ylabel("bps")
ax[0].legend(ncol=2, fontsize=9)
ax[0].set_title(f"Fair value definitions, last {len(tail)} bars "
                f"(horizon {res['horizon']} bars)")
 
ax[1].fill_between(tail.index, 0, p_hi, color="tab:orange", alpha=0.6)
ax[1].set_ylim(0, 1)
ax[1].set_ylabel("P(fast state)")
ax[1].set_xlabel("")
plt.tight_layout()
plt.show()

In [ ]:
from spread_cost import plot_pnl
 
dev = fv.deviation_series(f, res).dropna()
 
for name in ("static", "moving", "model"):
    print(f"\n######## FV definition: {name} ########")
    plot_pnl(dev[name].iloc[-2_000:], taker=0.05, maker=0.95,
             stop_bps=12, horizon_bars=600)

In [ ]:
# %% ------------------------------------------------------------------
# CELL 8 - the honesty check: how much of this is sample size?
#
# magnitude_check printed effective independent observations. Compare the
# difference in mean_net between rungs against its own standard error. If
# the gap is inside one standard error you have measured nothing.
# ---------------------------------------------------------------------
t = res["table"]
n_eff = len(f) * (1 - f["y"].autocorr(1)) / (1 + f["y"].autocorr(1))
for rung in t.index:
    if t.loc[rung, "n"] and np.isfinite(t.loc[rung, "mean_net"]):
        n_tr = t.loc[rung, "n"]
        # trades overlap at horizon h, so deflate the trade count too
        n_ind = max(n_tr / max(res["horizon"], 1), 1)
        se = np.nanstd(res["data"]["d"]) / np.sqrt(n_ind)
        print(f"{rung:<14} mean_net {t.loc[rung,'mean_net']:+.3f}  "
              f"se ~{se:.3f}  ({n_ind:.0f} independent trades)")
print(f"\neffective independent bars in sample: {n_eff:.0f} of {len(f)}")